# RetinaScan AI — EfficientNetB3 Transfer Learning

**Run this notebook on Google Colab with T4 GPU.**

What this notebook does:
1. Clones your GitHub repo (gets preprocessing + model code)
2. Downloads preprocessed images from your Kaggle dataset
3. Trains EfficientNetB3 in two phases
4. Evaluates and downloads the trained model

**Before running:**
- Runtime → Change runtime type → GPU (T4)
- Have your `kaggle.json` ready to upload

## Cell 1: Enable GPU Check

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f"GPU Available: {len(gpus) > 0}")
print(f"GPU Details: {gpus}")

if len(gpus) == 0:
    print("\n⚠️  WARNING: No GPU found!")
    print("Go to: Runtime → Change runtime type → GPU → Save")
    print("Then: Runtime → Restart runtime → Run all cells again")
else:
    print("\n✅ GPU ready! Proceeding...")

## Cell 2: Clone GitHub Repo

In [ ]:
import os

# ─── CHANGE THIS TO YOUR GITHUB USERNAME ───
GITHUB_USERNAME = "YOUR_GITHUB_USERNAME"
REPO_NAME = "Diabetic-Retinopathy"
# ───────────────────────────────────────────

# Remove old clone if exists
if os.path.exists(f'/content/{REPO_NAME}'):
    !rm -rf /content/{REPO_NAME}
    print("Removed old clone")

# Clone fresh
!git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git

# Navigate into repo
%cd /content/{REPO_NAME}

# Verify
print("\nRepo contents:")
!ls -la
print("\nSrc contents:")
!ls src/

## Cell 3: Install Libraries

In [ ]:
!pip install -q opencv-python-headless
!pip install -q albumentations
!pip install -q scikit-learn

print("✅ Libraries installed!")

## Cell 4: Setup Kaggle & Download Preprocessed Images

In [ ]:
from google.colab import files

# Upload kaggle.json
print("📤 Upload your kaggle.json file:")
uploaded = files.upload()

# Place it correctly
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("\n📥 Downloading preprocessed images from Kaggle...")
print("   This takes 2-3 minutes (much faster than uploading!)")

# Create data directory
!mkdir -p data/processed

# Download YOUR dataset
!kaggle datasets download -d jeetpaghdar/aptos-preprocessed-train -p data/processed/

print("\n📦 Unzipping...")
!unzip -q data/processed/aptos-preprocessed-train.zip -d data/processed/train_images/
!rm data/processed/aptos-preprocessed-train.zip

# Verify
count = len(os.listdir('data/processed/train_images/'))
print(f"\n✅ Done! {count} images ready")

## Cell 5: Download Original CSV (for labels)

In [ ]:
# Download APTOS CSV files for labels
!mkdir -p data/raw
!kaggle competitions download -c aptos2019-blindness-detection -p data/raw/ --file train.csv
!unzip -q data/raw/train.csv.zip -d data/raw/ 2>/dev/null || true

# Verify
import pandas as pd
train_csv = pd.read_csv('data/raw/train.csv')
print(f"✅ Loaded {len(train_csv)} labels")
print(train_csv['diagnosis'].value_counts().sort_index())

## Cell 6: Import All Libraries

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Add src to path
sys.path.append('src')
from model_utils import (
    build_efficientnet,
    compile_model,
    get_callbacks,
    evaluate_model,
    plot_training_history
)

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✅ All imports successful!")
print(f"TensorFlow version: {tf.__version__}")

## Cell 7: Prepare Dataset

In [ ]:
# Load CSV
train_csv = pd.read_csv('data/raw/train.csv')

# Point to PREPROCESSED images (not raw)
train_csv['image_path'] = train_csv['id_code'].apply(
    lambda x: f"data/processed/train_images/{x}.png"
)

# Keep only images that exist
train_csv['exists'] = train_csv['image_path'].apply(os.path.exists)
missing = (~train_csv['exists']).sum()
if missing > 0:
    print(f"⚠️  {missing} images missing, removing them")
train_csv = train_csv[train_csv['exists']].drop('exists', axis=1).reset_index(drop=True)

# Train / Validation split (80/20, stratified)
train_df, val_df = train_test_split(
    train_csv,
    test_size=0.20,
    stratify=train_csv['diagnosis'],
    random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"✅ Training samples  : {len(train_df)}")
print(f"✅ Validation samples: {len(val_df)}")
print(f"\nClass distribution (train):")
print(train_df['diagnosis'].value_counts().sort_index())

## Cell 8: Class Weights

In [ ]:
classes = np.unique(train_df['diagnosis'])
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_df['diagnosis']
)
class_weights = dict(zip(classes, weights))

print("⚖️  Class Weights:")
grade_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
for cls, weight in class_weights.items():
    print(f"  Grade {cls} ({grade_names[cls]:15s}): {weight:.4f}")

## Cell 9: Data Generator (Fast — loads preprocessed images directly)

In [ ]:
def create_generator(dataframe, batch_size, img_size=224, shuffle=True, augment=False):
    """
    Fast generator — loads already-preprocessed images.
    No on-the-fly preprocessing needed (that's what caused slowness before!)
    """
    num_samples = len(dataframe)
    num_classes = 5

    while True:
        if shuffle:
            dataframe = dataframe.sample(frac=1).reset_index(drop=True)

        for offset in range(0, num_samples, batch_size):
            batch_df = dataframe.iloc[offset:offset + batch_size]
            batch_images = []
            batch_labels = []

            for _, row in batch_df.iterrows():
                try:
                    # Load preprocessed image
                    img = cv2.imread(row['image_path'])
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = cv2.resize(img, (img_size, img_size))
                    img = img.astype(np.float32) / 255.0

                    # Augmentation (training only)
                    if augment:
                        if np.random.random() > 0.5:
                            img = np.fliplr(img)
                        if np.random.random() > 0.5:
                            img = np.flipud(img)
                        if np.random.random() > 0.5:
                            angle = np.random.uniform(-20, 20)
                            h, w = img.shape[:2]
                            M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
                            img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)

                    batch_images.append(img)
                    batch_labels.append(row['diagnosis'])

                except Exception as e:
                    continue

            if len(batch_images) > 0:
                X = np.array(batch_images, dtype=np.float32)
                y = to_categorical(batch_labels, num_classes=num_classes)
                yield X, y


# Config
IMG_SIZE   = 224
BATCH_SIZE = 32

train_gen = create_generator(train_df, BATCH_SIZE, IMG_SIZE, shuffle=True,  augment=True)
val_gen   = create_generator(val_df,   BATCH_SIZE, IMG_SIZE, shuffle=False, augment=False)

steps_per_epoch  = len(train_df) // BATCH_SIZE
validation_steps = len(val_df)   // BATCH_SIZE

print(f"✅ Generators ready!")
print(f"   Steps per epoch  : {steps_per_epoch}")
print(f"   Validation steps : {validation_steps}")

# Quick test
X_test, y_test = next(train_gen)
print(f"   Batch shape      : {X_test.shape}")
print(f"   Pixel range      : [{X_test.min():.2f}, {X_test.max():.2f}]")

## Cell 10: Visualize Sample Images

In [ ]:
sample_gen = create_generator(train_df, batch_size=10, img_size=IMG_SIZE, augment=False)
X_samples, y_samples = next(sample_gen)

grade_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_samples[i])
    grade = np.argmax(y_samples[i])
    ax.set_title(f'Grade {grade}: {grade_names[grade]}', fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sample Preprocessed Retinal Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('models/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Images look good — preprocessing confirmed working!")

## Cell 11: Build EfficientNetB3 — Phase 1 (Frozen Base)

In [ ]:
os.makedirs('models', exist_ok=True)

print("🏗️  Building EfficientNetB3 (Phase 1 — frozen base)...")
model = build_efficientnet(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    num_classes=5,
    dropout_rate=0.3,
    trainable_base=False      # Frozen for Phase 1
)
model = compile_model(model, learning_rate=1e-3)
model.summary()

## Cell 12: Train Phase 1 (~45 mins)

In [ ]:
callbacks_p1 = get_callbacks(
    model_save_path='models/efficientnet_phase1.keras',
    patience_early_stop=6,
    patience_lr=3
)

print("\n" + "="*60)
print("  PHASE 1 TRAINING — Frozen base, LR=0.001")
print("  Expected time: ~45 minutes on T4 GPU")
print("="*60 + "\n")

history1 = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=15,
    validation_data=val_gen,
    validation_steps=validation_steps,
    class_weight=class_weights,
    callbacks=callbacks_p1,
    verbose=1
)

print(f"\n✅ Phase 1 complete!")
print(f"   Best val_accuracy: {max(history1.history['val_accuracy']):.4f}")

## Cell 13: Fine-tune Phase 2 (Partial Unfreeze, ~30 mins)

In [ ]:
print("🔓 Unfreezing top 30 layers for fine-tuning...")

# Rebuild with partial unfreeze
model = build_efficientnet(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    num_classes=5,
    dropout_rate=0.3,
    trainable_base=True       # Partially unfrozen for Phase 2
)

# Load best weights from Phase 1
model.load_weights('models/efficientnet_phase1.keras')
print("✅ Phase 1 weights loaded")

# Compile with MUCH lower learning rate
model = compile_model(model, learning_rate=1e-5)

callbacks_p2 = get_callbacks(
    model_save_path='models/efficientnet_best.keras',
    patience_early_stop=5,
    patience_lr=3
)

print("\n" + "="*60)
print("  PHASE 2 FINE-TUNING — Partial unfreeze, LR=0.00001")
print("  Expected time: ~30 minutes on T4 GPU")
print("="*60 + "\n")

# Recreate generators (they get exhausted)
train_gen = create_generator(train_df, BATCH_SIZE, IMG_SIZE, shuffle=True,  augment=True)
val_gen   = create_generator(val_df,   BATCH_SIZE, IMG_SIZE, shuffle=False, augment=False)

history2 = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=10,
    validation_data=val_gen,
    validation_steps=validation_steps,
    class_weight=class_weights,
    callbacks=callbacks_p2,
    verbose=1
)

print(f"\n✅ Phase 2 complete!")
print(f"   Best val_accuracy: {max(history2.history['val_accuracy']):.4f}")

## Cell 14: Plot Training History

In [ ]:
plot_training_history(
    history1,
    history2,
    model_name='EfficientNetB3',
    save_path='models/efficientnet_history.png'
)

## Cell 15: Evaluate Final Model

In [ ]:
# Load best model
best_model = keras.models.load_model('models/efficientnet_best.keras')

# Recreate val generator
val_gen_eval = create_generator(val_df, BATCH_SIZE, IMG_SIZE, shuffle=False, augment=False)

# Evaluate
results = evaluate_model(best_model, val_gen_eval, validation_steps)

# Save results
import json
save_results = {k: float(v) for k, v in results.items() if k not in ['y_true', 'y_pred', 'y_pred_probs']}
with open('models/efficientnet_results.json', 'w') as f:
    json.dump(save_results, f, indent=2)
print("\n✅ Results saved to models/efficientnet_results.json")

## Cell 16: Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(results['y_true'], results['y_pred'])
grade_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Prolif.']

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=grade_names,
    yticklabels=grade_names
)
plt.title('EfficientNetB3 — Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('models/efficientnet_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## Cell 17: Download Everything

In [ ]:
from google.colab import files

print("💾 Downloading trained model and results...")
print("   (Files will appear in your Downloads folder)\n")

files.download('models/efficientnet_best.keras')
files.download('models/efficientnet_history.png')
files.download('models/efficientnet_confusion_matrix.png')
files.download('models/efficientnet_results.json')

print("✅ All files downloaded!")
print("\nSave these to your local project:")
print("  efficientnet_best.keras          → models/")
print("  efficientnet_history.png         → models/")
print("  efficientnet_confusion_matrix.png→ models/")
print("  efficientnet_results.json        → models/")

## Cell 18: Final Summary

In [ ]:
print("\n" + "="*60)
print("  EFFICIENTNETB3 TRAINING COMPLETE!")
print("="*60)
print(f"\n  Accuracy : {results['accuracy']*100:.2f}%")
print(f"  F1 Score : {results['f1']:.4f}")
print(f"  Kappa    : {results['kappa']:.4f}")
print("\n  Next step: Run 05_resnet_training.ipynb")
print("  (Same process, different architecture)")
print("="*60)